# Pipeline de Features con Feast

_De la ingenieria de features en un notebook a un feature store productivo: offline/online, point-in-time y materializacion._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Feature Store con Feast](assets/header.png)


## Introduccion

En el Notebook 1 hicimos ingenieria de features *dentro* de un notebook. Eso
sirve para experimentar, pero se rompe en produccion:

- La transformacion que corrio en entrenamiento debe correr **identica** en
  serving, o aparece el **sesgo entrenamiento-serving** (*training-serving skew*).
- Un set de entrenamiento debe usar **solo informacion disponible en el momento de
  cada evento** (sin espiar el futuro): **correctitud point-in-time**.
- Los equipos reimplementan las mismas features una y otra vez en lugar de
  **reutilizarlas**.

Un **feature store** resuelve esto. Usamos [Feast](https://feast.dev).


## Que es un feature store

Un feature store es la interfaz entre los datos crudos y los modelos. Piensa en
el como una *biblioteca de features*: alguien define una feature una vez y
cualquier equipo la consume por su nombre, con la garantia de que se calcula
igual en todas partes. Tiene dos capas de almacenamiento:

**Offline store** — el historico completo y *con marca de tiempo* de cada feature
(parquet, BigQuery, Snowflake...). Se usa para construir **datasets de
entrenamiento** con joins point-in-time.

**Online store** — solo el *ultimo* valor por entidad, en un almacen clave-valor
de baja latencia (Redis, DynamoDB...). Se usa para **serving online** con latencia
de milisegundos.

```
   datos crudos -> feature engineering -> OFFLINE store (parquet, historia completa)
                                               |  feast materialize
                                               v
                                          ONLINE store (Redis, ultimo/entidad)
   get_historical_features <-----------------+--------------> get_online_features
        (entrenamiento)                       v                (serving)
                            REGISTRY (registry.db / Postgres)
```

### Conceptos clave

- **Correctitud point-in-time.** `get_historical_features` une el
  `event_timestamp` de cada etiqueta contra la historia de features y devuelve el
  valor que era valido *en o antes* de ese momento, nunca uno del futuro. Esto
  evita la fuga de etiqueta (*label leakage*). Formalmente, para un evento en el
  instante $t$ devuelve $f(\text{entidad},\,\max\{s \le t\})$ dentro del TTL de la
  feature.
- **Sesgo entrenamiento-serving.** Como *las mismas definiciones de features*
  alimentan tanto `get_historical_features` (train) como `get_online_features`
  (serve), la transformacion no puede divergir entre ambos caminos.
- **Materializacion.** `feast materialize` copia los ultimos valores del offline
  store al online store para que el serving sea rapido.
- **Reutilizacion.** Las features se definen una vez en un **registry** y
  cualquier equipo/modelo las consume por su nombre.


El diagrama de abajo (dibujado con matplotlib) resume la arquitectura: las
fuentes de datos alimentan el **offline store**, `materialize` empuja el ultimo
valor al **online store**, el **registry** cataloga las definiciones, y hay dos
caminos de lectura segun el caso de uso.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 8); ax.axis("off")

def caja(x, y, w, h, texto, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                fc=color, ec="#333", lw=1.4))
    ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=10)

def flecha(p1, p2, texto="", color="#333", off=0.15):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>",
                                 mutation_scale=18, color=color, lw=1.6))
    if texto:
        mx, my = (p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2
        ax.text(mx, my + off, texto, ha="center", fontsize=9,
                color=color, style="italic")

# Fuentes -> offline store
caja(0.3, 5.2, 2.4, 1.4, "Fuentes de datos\n(raw, ETL)", "#cfe8ff")
caja(3.6, 5.2, 3.0, 1.4, "OFFLINE store\n(parquet)\nhistoria completa", "#d6f5d6")
flecha((2.7, 5.9), (3.6, 5.9), "feature\nengineering", off=0.35)

# Offline -> online (materialize)
caja(8.0, 5.2, 3.4, 1.4, "ONLINE store\n(Redis)\nultimo valor/entidad", "#ffe2b3")
flecha((6.6, 5.9), (8.0, 5.9), "materialize", off=0.2)

# Registry
caja(3.6, 2.9, 3.0, 1.2, "REGISTRY\n(catalogo de\nentidades / views)", "#f3d1ff")
flecha((5.1, 5.2), (5.1, 4.1), color="#888")

# Read paths
caja(0.3, 0.4, 3.6, 1.4,
     "get_historical_features\n(entrenamiento, point-in-time)", "#e8e8e8")
caja(7.6, 0.4, 3.8, 1.4,
     "get_online_features\n(serving, ms latencia)", "#e8e8e8")
flecha((4.6, 5.2), (2.1, 1.8), "lee historia", color="#1f77b4", off=0.3)
flecha((9.7, 5.2), (9.5, 1.8), "lee ultimo valor", color="#d62728", off=0.3)

ax.text(6, 7.4, "Arquitectura de un feature store (Feast)",
        ha="center", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## La estructura del repo de Feast

Nuestro repo vive ahora en la **plataforma compartida**, en
`../platform/feature_repo/`:

```
feature_repo/
├── feature_store.yaml   # config: provider, online_store (redis), registry, offline_store
├── features.py          # definiciones de Entity + FileSource + FeatureView
└── data/                # titanic_features.parquet (fuente) + registry.db
```

Los bloques de construccion (API actual de Feast, `feast>=0.40`):

- **`Entity`** — la clave de join a la que se cuelgan las features (aqui
  `passenger_id`).
- **`FileSource`** — apunta al parquet, con una columna `event_timestamp` para los
  joins point-in-time.
- **`Field`** (con tipos de `feast.types`) — una columna de feature declarada.
- **`FeatureView`** — un grupo nombrado y reutilizable de `Field`s atado a una
  entidad y a una fuente, con un **TTL** que acota cuan vieja puede ser un valor.

Ver [`../platform/feature_repo/features.py`](../platform/feature_repo/features.py).


## Paso 1 — ingenieria de features y escribir el parquet offline

Reconstruimos algunas features del Titanic y las escribimos en el parquet que el
`FileSource` espera. Feast requiere en cada fila una **columna de entidad** y una
columna **`event_timestamp`**.

In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

np.random.seed(0)

REPO_DATA = os.path.abspath(os.path.join("..", "platform", "feature_repo", "data"))
os.makedirs(REPO_DATA, exist_ok=True)
PARQUET_PATH = os.path.join(REPO_DATA, "titanic_features.parquet")

def load_titanic():
    try:
        import seaborn as sns
        d = sns.load_dataset("titanic")
        if d is None or len(d) == 0:
            raise ValueError("vacio")
        print("Dataset real del Titanic cargado.")
        return d
    except Exception as e:
        print(f"Fallback sintetico ({type(e).__name__}).")
        n = 400
        return pd.DataFrame({
            "pclass": np.random.choice([1, 2, 3], n),
            "sex": np.random.choice(["male", "female"], n),
            "age": np.clip(np.random.normal(29, 14, n), 0.4, 80).round(1),
            "sibsp": np.random.poisson(0.5, n),
            "parch": np.random.poisson(0.4, n),
            "fare": np.round(np.random.exponential(32, n), 4),
            "embark_town": np.random.choice(
                ["Southampton", "Cherbourg", "Queenstown"], n),
        })

df = load_titanic()
df.head(3)

In [ ]:
from sklearn.preprocessing import StandardScaler

feat = pd.DataFrame()
feat["passenger_id"] = np.arange(len(df), dtype="int64")   # la CLAVE de entidad

feat["age"] = df["age"].fillna(df["age"].median()).astype("float32")
feat["fare"] = df["fare"].fillna(df["fare"].median()).astype("float32")
feat["fare_log"] = np.log1p(feat["fare"]).astype("float32")        # domestica el sesgo
feat["age_scaled"] = StandardScaler().fit_transform(
    feat[["age"]]).astype("float32")                               # z-score
feat["family_size"] = (df["sibsp"].fillna(0) + df["parch"].fillna(0)
                       + 1).astype("int64")
feat["is_alone"] = (feat["family_size"] == 1).astype("int64")
feat["pclass"] = df["pclass"].fillna(3).astype("int64")
feat["sex"] = df["sex"].fillna("unknown").astype(str)
feat["embark_town"] = df["embark_town"].fillna("unknown").astype(str)

# Columnas de control de Feast: cuando ocurrio el evento y cuando se creo la fila.
now = datetime.utcnow()
feat["event_timestamp"] = [now - timedelta(days=int(i % 30))
                           for i in range(len(feat))]
feat["created"] = now

feat.head()

In [ ]:
feat.to_parquet(PARQUET_PATH, index=False)
print("Escritas", len(feat), "filas ->", PARQUET_PATH)
print("Columnas:", list(feat.columns))
pd.read_parquet(PARQUET_PATH).dtypes

## Paso 2 — `feast apply`

`feast apply` parsea `features.py`, lo valida y escribe las definiciones de
entidad / feature-view / fuente en el **registry**. Se ejecuta desde el
directorio del repo de features.

> Requiere `pip install "feast[redis]"` (el extra `[redis]` trae el driver del
> online store de Redis que configura el `feature_store.yaml` de este repo). Si
> Feast no esta instalado, la celda imprime el comando para que lo corras en una
> terminal en lugar de romper el notebook.

In [ ]:
import subprocess, shutil, sys

REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))
print("repo de features:", REPO)

def feast(*args):
    """Corre un comando del CLI de feast dentro del repo, mostrando la salida."""
    if shutil.which("feast") is None:
        print(f"[omitido] feast no instalado. Corre:\n"
              f"    cd {REPO} && feast {' '.join(args)}")
        return None
    res = subprocess.run(["feast", *args], cwd=REPO,
                         capture_output=True, text=True)
    print(res.stdout or "", res.stderr or "")
    return res

feast("apply")

## Paso 3 — features historicas (point-in-time) para entrenar

`get_historical_features` toma un **dataframe de entidades** (las filas con las
que quieres entrenar, cada una con su clave de entidad y un `event_timestamp`,
tipicamente el momento de la etiqueta) y devuelve un join point-in-time-correcto:
para cada fila trae el valor de feature que era valido *en o antes* de ese
timestamp, dentro del TTL.

Este es el camino de **entrenamiento**.

In [ ]:
try:
    from feast import FeatureStore
    store = FeatureStore(repo_path=REPO)

    # Dataframe de entidades: que pasajeros y en que momento traer features.
    entity_df = pd.DataFrame({
        "passenger_id": [0, 1, 2, 5, 10],
        "event_timestamp": [datetime.utcnow()] * 5,
    })

    training_df = store.get_historical_features(
        entity_df=entity_df,
        features=[
            "titanic_passenger_features:age_scaled",
            "titanic_passenger_features:fare_log",
            "titanic_passenger_features:family_size",
            "titanic_passenger_features:pclass",
            "titanic_passenger_features:sex",
        ],
    ).to_df()
    print("Frame de entrenamiento point-in-time:")
    display(training_df)
except Exception as e:
    print("get_historical_features omitido:", type(e).__name__, e)
    print("Asegurate de que `feast apply` funciono y de que el parquet existe.")

## Paso 4 — materializar offline → online

`materialize-incremental` carga los ultimos valores de feature en el **online
store (Redis)** para servirlos con latencia de milisegundos. Primero arranca
Redis (ahora vive en la plataforma compartida):

```bash
cd ../platform && docker compose up -d redis
```

El `feature_store.yaml` de este repo apunta el online store a `localhost:6379`.

In [ ]:
from datetime import datetime as _dt
end = _dt.utcnow().strftime("%Y-%m-%dT%H:%M:%S")
# Carga todo hasta `end` en Redis. Necesita el contenedor redis corriendo.
feast("materialize-incremental", end)

## Paso 5 — features online para serving

`get_online_features` lee el **ultimo** valor por entidad desde Redis. Este es el
camino de **serving**: ojo que usa los *mismos nombres de feature* que el camino
de entrenamiento, que es justo lo que mata el sesgo entrenamiento-serving.

In [ ]:
try:
    online = store.get_online_features(
        features=[
            "titanic_passenger_features:age_scaled",
            "titanic_passenger_features:fare_log",
            "titanic_passenger_features:family_size",
            "titanic_passenger_features:pclass",
        ],
        entity_rows=[{"passenger_id": 0}, {"passenger_id": 5}],
    ).to_dict()
    display(pd.DataFrame(online))
except Exception as e:
    print("get_online_features omitido:", type(e).__name__, e)
    print("Necesita: redis corriendo (docker compose up -d) + feast materialize hecho.")

## Como se conecta con Docker

`../platform/docker-compose.yml` levanta los servicios de soporte (y, ademas,
Airflow como orquestador compartido del curso):

| Servicio | Imagen | Rol | Puerto |
|---|---|---|---|
| `redis` | `redis:7` | **online store** (serving) | 6379 |
| `postgres` | `postgres:16` | **registry / offline** opcional | 5432 |

`feature_store.yaml` conecta Feast con ellos:

```yaml
online_store:
  type: redis
  connection_string: "localhost:6379"   # -> el contenedor redis
registry: data/registry.db               # registry en archivo (por defecto)
# o cambia al registry `sql` de Postgres (-> el contenedor postgres)
```

De punta a punta:

```bash
cd ../platform
docker compose up -d redis postgres       # arranca redis + postgres
cd feature_repo
feast apply                               # registra las definiciones
feast materialize-incremental $(date +%Y-%m-%dT%H:%M:%S)   # offline -> Redis
# ... luego get_online_features sirve desde Redis
```

Ver [`../platform/README.md`](../platform/README.md) para la referencia completa
de la plataforma compartida (Feast + Airflow).

## Repaso

- Un **feature store** desacopla la *definicion* de features de su *consumo*,
  dandote reutilizacion, datos de entrenamiento point-in-time correctos y serving
  sin sesgo.
- **Offline store** = historia completa con timestamps (parquet) → entrenamiento
  via `get_historical_features`.
- **Online store** = ultimo valor por entidad (Redis) → serving via
  `get_online_features`.
- `feast apply` registra definiciones; `feast materialize` empuja offline →
  online; los **mismos nombres de feature** manejan entrenamiento y serving.

En el **Notebook 4** cerramos el ciclo: tomamos estas features, construimos el set
de entrenamiento, entrenamos un modelo y lo evaluamos.